In [61]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import re

In [62]:
fake_df=pd.read_csv('Fake.csv')
true_df=pd.read_csv('True.csv')

In [63]:
fake_df.head()
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [64]:
fake_df.info()
true_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [65]:
fake_df ['label'] =0     #fake news
true_df ['label'] =1     #Real news

In [66]:
#combining the datasets

In [67]:
df = pd.concat([fake_df, true_df], axis=0)

In [68]:
#Reshuttling the dataset

In [69]:
df=df.sample(frac=1, random_state=46).reset_index(drop=True)

In [70]:
df.shape
df.columns

Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')

In [71]:
#Checking of missing value


In [72]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [73]:
#checking of duplicate

In [74]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [75]:
df['content']= df['title'] + " "+ ['text']

In [76]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
#Removing of stopwords

In [77]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df['content'] = df['content'].apply(remove_stopwords)

In [ ]:
#converting text to numbers

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=15000)

X = vectorizer.fit_transform(df['content'])
y = df['label']

In [83]:
#Checking of Class distribution

sns.countplot(x=df['label'])
plt.title("Fake (0) vs Real (1)")
plt.show()

In [85]:
#Spliting of the dataset

In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [88]:
#Training the MOdel using Support Vector Machine

In [93]:
from sklearn.svm import SVC
svm_model= SVC()
svm_model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [91]:
#Making Prediction 

In [101]:
svm_pred=svm_model.predict(X_test)

In [102]:
#Train model 2 using Random Forest

In [103]:
from sklearn.ensemble import RandomForestClassifier   
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)



,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [104]:
rf_pred = rf_model.predict(X_test)

In [105]:
#Evaluating the model 

In [110]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("===== SVM RESULTS =====")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))

print("\n===== RANDOM FOREST RESULTS =====")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))

===== SVM RESULTS =====
Accuracy: 0.9866487655702245
Precision: 0.9792312461252325
Recall: 0.9929278642149929
F1 Score: 0.9860319937573157

===== RANDOM FOREST RESULTS =====
Accuracy: 0.9831431341836354
Precision: 0.988070621918244
Recall: 0.9762690554769763
F1 Score: 0.9821343873517786


## Model Evaluation Results

The Support Vector Machine (SVM) and Random Forest models were evaluated using Accuracy, Precision, Recall, and F1-score. SVM achieved slightly better overall performance, especially in recall and F1-score, making it more suitable for fake news detection using TF-IDF features.